# Aim 1 + Aim 3 Follow-Up Validation Suite (Colab Pro High-RAM)
**Two High-Impact Reviewer-Proof Modules (Automated Live Streaming + 5 TB Drive Cache):**
1. **Module A (Cells [1]–[3])**: Single-Cell Consensus Cell-Cell Communication (**LIANA+** `rank_aggregate`: CellPhoneDB + NATMI + Connectome + SingleCellSignalR) comparing **37 MDD vs 35 Control donors** (`InN1_PV` / `InN5_SST` / `InN4_VIP` → `ExN2_L23` / `ExN3_L46` / `ExN10_L46`) across 120,578 DLPFC single nuclei.
2. **Module B (Cells [4]–[5])**: **Bryois et al. 2022 (Nature Neuroscience) 8-Brain-Cell-Type cis-eQTL Colocalization** (`Excitatory.neurons`, `Inhibitory.neurons`, `Astrocytes`, `Oligodendrocytes`, `OPCs...COPs`, `Microglia`, `Endothelial.cells`, `Pericytes` from Zenodo `7276971` / DOI `10.5281/zenodo.5543734`) against million-person **MDD ($N=1.15	ext{M}$)** & **ANX ($N=1.10	ext{M}$)** GWAS across all 7 dual-trait fine-mapped loci (`SLC12A5` chr20, `DCC` chr18, `PPP6C` chr9, `chr3p21`, `DNAH10/CDK2AP1` chr12, `SLC25A12` chr2).

> **Runtime Tip:** Set Runtime -> Change runtime type -> **High-RAM CPU**. Outputs are permanently archived into your Google Drive (`01_Scheme1_colab_outputs` & `03_aim2_aim3_outputs`).

In [ ]:
# [1] Install Python (LIANA+, Scanpy) & Linux R (coloc, data.table) + Mount 5 TB Drive
import subprocess, sys, os, psutil

def run_live(cmd_list, desc=''):
    if desc:
        print(desc, flush=True)
    p = subprocess.Popen(cmd_list, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd_list)

print('[Step 1A] Installing Python packages (scanpy, anndata, harmonypy, liana, psutil)...', flush=True)
run_live([sys.executable, '-m', 'pip', 'install', '-q', 'scanpy', 'anndata', 'harmonypy', 'liana', 'psutil'])

print('[Step 1B] Checking / installing R & coloc for Bryois 8-cell-type colocalization...', flush=True)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'plink1.9', 'unzip', 'r-base', 'r-cran-data.table', 'r-cran-remotes'], check=True)
run_live(['Rscript', '-e', "if (!requireNamespace('coloc', quietly=TRUE)) { options(repos=c(CRAN='https://cloud.r-project.org')); install.packages('coloc', quiet=TRUE) }; cat('R coloc ready!\n')"])

from google.colab import drive
print('[Step 1C] Mounting Google Drive...', flush=True)
drive.mount('/content/drive')

DRIVE_ROOT   = '/content/drive/MyDrive/MDD_New_Project_Pipelines_2026'
DRIVE_IN_S1  = os.path.join(DRIVE_ROOT, '01_Scheme1_inputs')
DRIVE_OUT_S1 = os.path.join(DRIVE_ROOT, '01_Scheme1_colab_outputs')
DRIVE_FIGS   = os.path.join(DRIVE_OUT_S1, 'figures')
DRIVE_CACHE  = os.path.join(DRIVE_ROOT, '03_drive_cache')
DRIVE_IN_A2  = os.path.join(DRIVE_ROOT, '03_aim2_inputs')
DRIVE_OUT_A3 = os.path.join(DRIVE_ROOT, '03_aim2_aim3_outputs')
WORK_DIR     = '/content/followup_work'

for d in [DRIVE_IN_S1, DRIVE_OUT_S1, DRIVE_FIGS, os.path.join(DRIVE_CACHE, 'bryois_zenodo_5543734'), DRIVE_OUT_A3, os.path.join(WORK_DIR, 'inputs'), os.path.join(WORK_DIR, 'loci_dualtrait')]:
    os.makedirs(d, exist_ok=True)

total_ram = psutil.virtual_memory().total / (1024**3)
print(f'[SUCCESS] Environment ready! System RAM: {total_ram:.1f} GB', flush=True)


In [ ]:
# [2] Load or Rebuild Clean GSE213982 AnnData (120,578 QC Nuclei, Cached to Drive)
import os, shutil, gc, numpy as np, pandas as pd, scipy.io as sio, scanpy as sc, anndata as ad

INPUT_DIR = os.path.join(WORK_DIR, 'inputs')
h5ad_drive = os.path.join(DRIVE_OUT_S1, 'GSE213982_author41_clean.h5ad')
h5ad_local = os.path.join(WORK_DIR, 'GSE213982_author41_clean.h5ad')

CLINICAL_META_TSV = '''donor_id	geo_accession	sex	diagnosis	tissue
F1	GSM6597355	Female	MDD	Brain BA9 dlPFC
F10	GSM6597356	Female	Control	Brain BA9 dlPFC
F11	GSM6597357	Female	MDD	Brain BA9 dlPFC
F12	GSM6597358	Female	MDD	Brain BA9 dlPFC
F13	GSM6597359	Female	Control	Brain BA9 dlPFC
F14	GSM6597360	Female	MDD	Brain BA9 dlPFC
F15	GSM6597361	Female	MDD	Brain BA9 dlPFC
F16	GSM6597362	Female	MDD	Brain BA9 dlPFC
F17	GSM6597363	Female	MDD	Brain BA9 dlPFC
F18	GSM6597364	Female	MDD	Brain BA9 dlPFC
F19	GSM6597365	Female	MDD	Brain BA9 dlPFC
F2	GSM6597366	Female	MDD	Brain BA9 dlPFC
F20	GSM6597367	Female	MDD	Brain BA9 dlPFC
F21	GSM6597368	Female	Control	Brain BA9 dlPFC
F22	GSM6597369	Female	Control	Brain BA9 dlPFC
F23	GSM6597370	Female	Control	Brain BA9 dlPFC
F24	GSM6597371	Female	Control	Brain BA9 dlPFC
F25	GSM6597372	Female	MDD	Brain BA9 dlPFC
F26	GSM6597373	Female	Control	Brain BA9 dlPFC
F27	GSM6597374	Female	MDD	Brain BA9 dlPFC
F28	GSM6597375	Female	MDD	Brain BA9 dlPFC
F29	GSM6597376	Female	Control	Brain BA9 dlPFC
F3	GSM6597377	Female	MDD	Brain BA9 dlPFC
F30	GSM6597378	Female	Control	Brain BA9 dlPFC
F31	GSM6597379	Female	Control	Brain BA9 dlPFC
F32	GSM6597380	Female	Control	Brain BA9 dlPFC
F33	GSM6597381	Female	Control	Brain BA9 dlPFC
F34	GSM6597382	Female	Control	Brain BA9 dlPFC
F35	GSM6597383	Female	Control	Brain BA9 dlPFC
F36	GSM6597384	Female	Control	Brain BA9 dlPFC
F37	GSM6597385	Female	Control	Brain BA9 dlPFC
F38	GSM6597386	Female	Control	Brain BA9 dlPFC
F4	GSM6597387	Female	MDD	Brain BA9 dlPFC
F5	GSM6597388	Female	MDD	Brain BA9 dlPFC
F6	GSM6597389	Female	MDD	Brain BA9 dlPFC
F7	GSM6597390	Female	Control	Brain BA9 dlPFC
F8	GSM6597391	Female	MDD	Brain BA9 dlPFC
F9	GSM6597392	Female	MDD	Brain BA9 dlPFC
M1	GSM4281974	Male	MDD	Brain
M2	GSM4281975	Male	Control	Brain
M3	GSM4281976	Male	Control	Brain
M4	GSM4281977	Male	MDD	Brain
M5	GSM4281978	Male	MDD	Brain
M6	GSM4281979	Male	MDD	Brain
M7	GSM4281980	Male	Control	Brain
M8	GSM4281981	Male	MDD	Brain
M9	GSM4281982	Male	Control	Brain
M10	GSM4281983	Male	MDD	Brain
M11	GSM4281984	Male	MDD	Brain
M12	GSM4281985	Male	Control	Brain
M13	GSM4281986	Male	Control	Brain
M14	GSM4281987	Male	MDD	Brain
M15	GSM4281988	Male	Control	Brain
M16	GSM4281989	Male	Control	Brain
M17	GSM4281990	Male	MDD	Brain
M18	GSM4281991	Male	MDD	Brain
M19	GSM4281992	Male	Control	Brain
M20	GSM4281993	Male	Control	Brain
M21	GSM4281994	Male	Control	Brain
M22	GSM4281995	Male	Control	Brain
M23	GSM4281996	Male	MDD	Brain
M24	GSM4281997	Male	Control	Brain
M25	GSM4281998	Male	Control	Brain
M26	GSM4281999	Male	MDD	Brain
M27	GSM4282000	Male	Control	Brain
M28	GSM4282001	Male	MDD	Brain
M29	GSM4282002	Male	Control	Brain
M30	GSM4282003	Male	MDD	Brain
M31	GSM4282004	Male	Control	Brain
M32	GSM4282005	Male	MDD	Brain
M33	GSM4282006	Male	MDD	Brain
M34	GSM4282007	Male	MDD	Brain'''

if os.path.exists(h5ad_drive):
    print('[Step 2] Found cached clean .h5ad in Drive! Copying to NVMe...', flush=True)
    shutil.copy2(h5ad_drive, h5ad_local)
    adata = sc.read_h5ad(h5ad_local)
else:
    print('[Step 2A] Staging 10x raw counts from Drive to NVMe...', flush=True)
    for fname in ['GSE213982_combined_counts_matrix.mtx.gz', 'GSE213982_combined_counts_matrix_cells_columns.csv.gz', 'GSE213982_combined_counts_matrix_genes_rows.csv.gz']:
        src, dst = os.path.join(DRIVE_IN_S1, fname), os.path.join(INPUT_DIR, fname)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
    meta_p = os.path.join(INPUT_DIR, 'GSE213982_donor_clinical_metadata.tsv')
    with open(meta_p, 'w') as f:
        f.write(CLINICAL_META_TSV.strip() + '\n')

    print('[Step 2B] Reading MatrixMarket counts & QC filtering...', flush=True)
    X = sio.mmread(os.path.join(INPUT_DIR, 'GSE213982_combined_counts_matrix.mtx.gz')).T.tocsr()
    genes = pd.read_csv(os.path.join(INPUT_DIR, 'GSE213982_combined_counts_matrix_genes_rows.csv.gz'), index_col=0)
    cells = pd.read_csv(os.path.join(INPUT_DIR, 'GSE213982_combined_counts_matrix_cells_columns.csv.gz'), index_col=0)
    adata = ad.AnnData(X=X, obs=cells, var=genes)
    adata.var_names_make_unique()
    adata = adata[~adata.obs_names.isin(['"x"', 'x'])].copy()
    adata.obs_names = adata.obs_names.map(lambda b: str(b).replace('"', ''))
    if 'x' in adata.obs.columns:
        adata.obs = adata.obs.drop(columns=['x'])

    tokens = pd.Series(adata.obs_names, index=adata.obs_names).str.split('.', expand=True)
    adata.obs['sample_id'] = tokens[0]
    adata.obs['donor_id'] = adata.obs['sample_id'].replace({'M24_2': 'M24'})
    adata.obs['author_major'] = tokens[2].astype('category')
    adata.obs['author_subtype'] = tokens[3].astype('category')
    adata.obs['sex'] = adata.obs['donor_id'].map(lambda s: 'Female' if str(s).startswith('F') else 'Male')
    meta_df = pd.read_csv(meta_p, sep='\t').set_index('donor_id')
    adata.obs['diagnosis'] = adata.obs['donor_id'].map(meta_df['diagnosis'])

    adata.var['mt'] = adata.var_names.str.contains('^MT-', case=False, regex=True)
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    adata = adata[(adata.obs['n_genes_by_counts'] > 500) & (adata.obs['n_genes_by_counts'] < 8000) & (adata.obs['total_counts'] > 1000)].copy()
    adata = adata[~adata.obs['author_subtype'].astype(str).str.contains('Mix', case=False)].copy()

    adata.obs['is_doublet'] = False
    for sid in adata.obs['sample_id'].unique():
        mask = (adata.obs['sample_id'] == sid).to_numpy()
        if mask.sum() > 80:
            sub_ad = adata[mask].copy()
            try:
                sc.pp.scrublet(sub_ad, verbose=False)
                adata.obs.loc[mask, 'is_doublet'] = sub_ad.obs['predicted_doublet'].values
            except Exception:
                pass
    adata = adata[~adata.obs['is_doublet']].copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.write_h5ad(h5ad_local)
    shutil.copy2(h5ad_local, h5ad_drive)
    print(f'  -> Clean AnnData archived to Drive: {h5ad_drive}', flush=True)

print(f'[SUCCESS] AnnData ready: {adata.n_obs:,} nuclei across {adata.obs["author_subtype"].nunique()} subtypes.', flush=True)


In [ ]:
# [3] Module A: Single-Cell LIANA+ Consensus Cell-Cell Communication (MDD vs Control)
import liana as li
import matplotlib.pyplot as plt

# Focus on subtypes with >= 200 cells in both MDD and Control to avoid sparse sampling noise
counts_by_diag = adata.obs.groupby(['author_subtype', 'diagnosis'], observed=False).size().unstack(fill_value=0)
valid_subs = counts_by_diag[(counts_by_diag['MDD'] >= 150) & (counts_by_diag['Control'] >= 150)].index.tolist()
ad_comm = adata[adata.obs['author_subtype'].isin(valid_subs)].copy()
ad_comm.obs['author_subtype'] = ad_comm.obs['author_subtype'].cat.remove_unused_categories()
print(f'[Step 3A] Running LIANA consensus rank_aggregate across {len(valid_subs)} high-coverage subtypes...', flush=True)

res_list = []
for diag_grp in ['Control', 'MDD']:
    print(f'  -> Running LIANA on {diag_grp} nuclei...', flush=True)
    ad_sub = ad_comm[ad_comm.obs['diagnosis'] == diag_grp].copy()
    li.mt.rank_aggregate(
        ad_sub,
        groupby='author_subtype',
        resource_name='consensus',
        expr_prop=0.05,
        min_cells=20,
        use_raw=False,
        verbose=True,
        inplace=True,
    )
    df_l = ad_sub.uns['liana_res'].copy()
    df_l['diagnosis'] = diag_grp
    res_list.append(df_l)

liana_all = pd.concat(res_list, ignore_index=True)
out_full = os.path.join(DRIVE_OUT_S1, 'liana_mdd_vs_control_full.csv')
liana_all.to_csv(out_full, index=False)

print('[Step 3B] Computing differential MDD vs Control interaction magnitude shifts...', flush=True)
ctrl_df = liana_all[liana_all['diagnosis'] == 'Control'][['source', 'target', 'ligand_complex', 'receptor_complex', 'magnitude_rank', 'specificity_rank', 'lr_means']]
mdd_df  = liana_all[liana_all['diagnosis'] == 'MDD'][['source', 'target', 'ligand_complex', 'receptor_complex', 'magnitude_rank', 'specificity_rank', 'lr_means']]

merged_lr = mdd_df.merge(ctrl_df, on=['source', 'target', 'ligand_complex', 'receptor_complex'], suffixes=('_MDD', '_Ctrl'))
merged_lr['delta_lr_means'] = merged_lr['lr_means_MDD'] - merged_lr['lr_means_Ctrl']
merged_lr['log2_fc_lr_means'] = np.log2((merged_lr['lr_means_MDD'] + 1e-4) / (merged_lr['lr_means_Ctrl'] + 1e-4))
merged_lr['delta_mag_rank'] = merged_lr['magnitude_rank_MDD'] - merged_lr['magnitude_rank_Ctrl']

# Focus on GABAergic Interneurons -> Pyramidal Excitatory Neurons
ei_edges = merged_lr[merged_lr['source'].str.contains('InN1_PV|InN5_SST|InN4_VIP') & merged_lr['target'].str.contains('ExN2_L23|ExN3_L46|ExN10_L46')].copy()
ei_edges = ei_edges.sort_values('log2_fc_lr_means')
out_ei = os.path.join(DRIVE_OUT_S1, 'liana_pv_sst_to_pyramidal_diff.csv')
ei_edges.to_csv(out_ei, index=False)

# Plot Top Differential Synaptic Interactions
top_loss = ei_edges.head(15)
top_gain = ei_edges.tail(10)
plot_df = pd.concat([top_loss, top_gain]).copy()
plot_df['label'] = plot_df['source'].astype(str) + ' -> ' + plot_df['target'].astype(str) + ' | ' + plot_df['ligand_complex'] + '-' + plot_df['receptor_complex']
plot_df = plot_df.sort_values('log2_fc_lr_means')

fig, ax = plt.subplots(figsize=(10, 8), dpi=300)
cols = ['#2166ac' if v < 0 else '#b2182b' for v in plot_df['log2_fc_lr_means']]
ax.barh(np.arange(len(plot_df)), plot_df['log2_fc_lr_means'], color=cols, alpha=0.88, edgecolor='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.9)
ax.set_yticks(np.arange(len(plot_df)))
ax.set_yticklabels(plot_df['label'], fontsize=8, fontweight='bold')
ax.set_xlabel('log2 Fold Change in LIANA Mean Interaction Strength (MDD / Control)', fontweight='bold')
ax.set_title('Differential InN -> ExN Synaptic Communication (LIANA+ Single-Cell)', loc='left', fontweight='bold', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_FIGS, 'Figure_Aim1_LIANA_MDD_vs_Control_Communication.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(DRIVE_FIGS, 'Figure_Aim1_LIANA_MDD_vs_Control_Communication.pdf'), bbox_inches='tight')
plt.show()
print(f'[SUCCESS] LIANA+ differential communication finished & saved to Drive!', flush=True)


In [ ]:
# [4] Module B: Auto-Stage Bryois 2022 8-Brain-Cell-Type eQTLs, Loci & 1000G EUR MAF
import os, glob, shutil, zipfile, urllib.request, json, subprocess, ssl

BRYOIS_DIR = os.path.join(DRIVE_CACHE, 'bryois_zenodo_5543734')
LD_DIR     = os.path.join(DRIVE_CACHE, 'ld_1kg_eur')
LOCI_WORK  = os.path.join(WORK_DIR, 'loci_dualtrait')
os.makedirs(BRYOIS_DIR, exist_ok=True)
os.makedirs(LD_DIR, exist_ok=True)
os.makedirs(LOCI_WORK, exist_ok=True)

# 4A. Auto-detect the 7 *_dualtrait_grch37.tsv locus files anywhere in Drive or Colab
found_loci = []
for root_dir in ['/content/drive/MyDrive', '/content']:
    for r, _, files in os.walk(root_dir):
        if '/content/drive/Shareddrives' in r:
            continue
        for fn in files:
            if fn.endswith('_dualtrait_grch37.tsv'):
                src = os.path.join(r, fn)
                dst = os.path.join(LOCI_WORK, fn)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
                found_loci.append(dst)

loci_files = sorted(set(found_loci))
print(f'[Step 4A] Staged {len(loci_files)} dual-trait locus files.', flush=True)
if len(loci_files) == 0:
    raise FileNotFoundError('Could not find *_dualtrait_grch37.tsv files in Drive. Ensure they are inside MyDrive.')

# 4B. Stage 1000G EUR Reference MAF (reuses Aim 2 cached zip in Drive!)
eur_zip = os.path.join(LD_DIR, 'g1000_eur.zip')
frq_out = os.path.join(WORK_DIR, 'g1000_eur.frq')
if not os.path.exists(frq_out):
    if not os.path.exists(eur_zip) or not zipfile.is_zipfile(eur_zip):
        print('[Step 4B] Downloading 1000G EUR LD panel (~511 MB)...', flush=True)
        ssl_ctx = ssl.create_default_context()
        ssl_ctx.check_hostname = False
        ssl_ctx.verify_mode = ssl.CERT_NONE
        req = urllib.request.Request('https://vu.data.surfsara.nl/index.php/s/VZNByNwpD8qqINe/download', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, context=ssl_ctx, timeout=180) as r, open(eur_zip, 'wb') as f_out:
            shutil.copyfileobj(r, f_out)
    else:
        print('[Step 4B] Reusing cached 1000G EUR panel from Drive!', flush=True)
    subprocess.run(['unzip', '-q', '-o', eur_zip, '-d', WORK_DIR], check=True)
    if not os.path.exists('/usr/local/bin/plink') and os.path.exists('/usr/bin/plink1.9'):
        os.symlink('/usr/bin/plink1.9', '/usr/local/bin/plink')
    print('  -> Computing 1000G EUR MAF table via PLINK...', flush=True)
    subprocess.run(['plink', '--bfile', os.path.join(WORK_DIR, 'g1000_eur'), '--freq', '--out', os.path.join(WORK_DIR, 'g1000_eur')], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
print(f'  -> 1000G EUR MAF ready: {frq_out}', flush=True)

# 4C. Download Bryois 2022 8-Brain-Cell-Type eQTL archives directly (Bypassing Zenodo API 403)
target_chrs = ('2', '3', '9', '12', '18', '20')
cell_types = ['Inhibitory.neurons', 'Excitatory.neurons', 'Astrocytes', 'Oligodendrocytes', 'OPCs...COPs', 'Microglia', 'Endothelial.cells', 'Pericytes']

total_needed = len(cell_types) * len(target_chrs)
print(f'[Step 4C] Staging Bryois 2022 8-brain-cell-type eQTLs for target chromosomes {target_chrs} ({total_needed} archives total)...', flush=True)

idx = 0
cached_cnt = 0
fetched_cnt = 0
for ct in cell_types:
    for ch in target_chrs:
        idx += 1
        fn = f'{ct}.{ch}.gz'
        dst = os.path.join(BRYOIS_DIR, fn)
        # If file exists and > 500 KB, reuse cache in Drive
        if os.path.exists(dst) and os.path.getsize(dst) > 500_000:
            cached_cnt += 1
            continue
        url = f'https://zenodo.org/records/7276971/files/{fn}'
        print(f'  [{idx}/{total_needed}] Downloading {fn} from Zenodo...', flush=True)
        ret = subprocess.run(['curl', '-f', '-sS', '-L', '--retry', '3', '--retry-delay', '2', '-o', dst, url], capture_output=True, text=True)
        if ret.returncode != 0 or not os.path.exists(dst) or os.path.getsize(dst) < 500_000:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
            with urllib.request.urlopen(req, timeout=90) as resp, open(dst, 'wb') as out_f:
                shutil.copyfileobj(resp, out_f)
        fetched_cnt += 1
        print(f'    -> Stored {fn} ({os.path.getsize(dst)/1024**2:.1f} MB)', flush=True)

print(f'[SUCCESS] Bryois eQTL archives ready: {cached_cnt} cached from Drive, {fetched_cnt} downloaded fresh!', flush=True)


In [ ]:
# [5] Module C: Cell-Type cis-eQTL Colocalization Across All 8 Brain Cell Types (MDD & ANX)
import subprocess, sys

r_bryois = rf"""
suppressPackageStartupMessages({{
  library(data.table)
  library(coloc)
}})

bryois_dir <- '{BRYOIS_DIR}'
loci_dir   <- '{LOCI_WORK}'
frq_path   <- '{os.path.join(WORK_DIR, "g1000_eur.frq")}'
out_dir    <- '{DRIVE_OUT_A3}'

cat('Loading 1000G EUR MAF reference...\n')
frq_dt <- fread(frq_path)[!duplicated(SNP), .(SNP, MAF = pmin(MAF, 1 - MAF))]

loci_files <- list.files(loci_dir, pattern='_dualtrait_grch37[.]tsv$', full.names=TRUE)
cell_types <- c('Inhibitory.neurons', 'Excitatory.neurons', 'Astrocytes', 'Oligodendrocytes', 'OPCs...COPs', 'Microglia', 'Endothelial.cells', 'Pericytes')
N_BRYOIS <- 192L

results <- list()

for (lf in loci_files) {{
  tag <- sub('_dualtrait_grch37[.]tsv$', '', basename(lf))
  gw <- fread(lf)
  gw <- merge(gw, frq_dt, by='SNP', all.x=FALSE)
  chr_val <- as.character(gw$CHR[1])
  cat(sprintf('\n>>> Processing Locus %s (chr%s, %d MAF-matched SNPs) <<<\n', tag, chr_val, nrow(gw)))
  flush.console()

  for (ct in cell_types) {{
    eq_f <- file.path(bryois_dir, sprintf('%s.%s.gz', ct, chr_val))
    if (!file.exists(eq_f)) next

    eq <- fread(cmd=sprintf('gzip -dc %s', shQuote(eq_f)), header=FALSE,
                col.names=c('gene_ensg', 'SNP', 'tss_dist', 'PVAL', 'BETA'))
    eq[, gene := sub('_ENSG.*', '', gene_ensg)]
    eq_sub <- eq[SNP %in% gw$SNP]
    if (nrow(eq_sub) == 0) next

    sig_genes <- eq_sub[, .(min_p = min(PVAL, na.rm=TRUE)), by=gene][min_p < 1e-3]$gene
    for (gname in sig_genes) {{
      eg <- eq_sub[gene == gname][!duplicated(SNP)]
      m <- merge(gw, eg, by='SNP')
      m <- m[!is.na(BETA_MDD) & SE_MDD > 0 & !is.na(BETA_ANX) & SE_ANX > 0 & !is.na(PVAL) & MAF > 0.01]
      if (nrow(m) < 50) next

      d_eqtl <- list(pvalues=pmax(as.numeric(m$PVAL), 1e-300), N=N_BRYOIS, MAF=m$MAF, snp=m$SNP, type='quant')

      for (tr in c('MDD', 'ANX')) {{
        b_tr  <- if (tr == 'MDD') m$BETA_MDD else m$BETA_ANX
        se_tr <- if (tr == 'MDD') m$SE_MDD   else m$SE_ANX
        n_tr  <- if (tr == 'MDD') median(m$N_MDD, na.rm=TRUE) else median(m$N_ANX, na.rm=TRUE)
        s_tr  <- if (tr == 'MDD') 0.34 else 0.25
        d_gwas <- list(beta=b_tr, varbeta=se_tr^2, N=n_tr, s=s_tr, MAF=m$MAF, snp=m$SNP, type='cc')

        suppressWarnings({{ res <- coloc.abf(dataset1=d_gwas, dataset2=d_eqtl) }})
        sm <- as.list(res$summary)
        lead_eq <- eg[which.min(PVAL)]
        results[[length(results)+1]] <- data.table(
          locus = tag, chr = chr_val, cell_type = ct, gene = gname, trait = tr,
          nsnps = sm$nsnps, PP.H3 = sm$PP.H3.abf, PP.H4 = sm$PP.H4.abf,
          top_eqtl_snp = lead_eq$SNP, top_eqtl_p = lead_eq$PVAL, top_eqtl_beta = lead_eq$BETA
        )
        if (sm$PP.H4.abf >= 0.50) {{
          cat(sprintf('  [COLOC HIT] %s | %s | %s vs %s: PP.H4=%.3f (top eSNP=%s, P=%.2e)\n',
                      ct, gname, tr, tag, sm$PP.H4.abf, lead_eq$SNP, lead_eq$PVAL))
          flush.console()
        }}
      }}
    }}
  }}
}}

if (length(results) > 0) {{
  final_dt <- rbindlist(results, fill=TRUE)[order(-PP.H4)]
  out_f <- file.path(out_dir, 'aim3_bryois_8celltype_coloc_MDD_and_ANX.tsv')
  fwrite(final_dt, out_f, sep='\t')
  cat(sprintf('\n[SUCCESS] Saved %d cell-type coloc tests to: %s\n', nrow(final_dt), out_f))
}} else {{
  cat('\nNo colocalization pairs met threshold.\n')
}}
"""
open('/content/run_bryois_celltype_coloc.R', 'w').write(r_bryois)

p = subprocess.Popen(['Rscript', '/content/run_bryois_celltype_coloc.R'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
p.wait()
if p.returncode != 0:
    raise subprocess.CalledProcessError(p.returncode, ['Rscript', '/content/run_bryois_celltype_coloc.R'])

import pandas as pd, numpy as np, matplotlib.pyplot as plt
out_f = os.path.join(DRIVE_OUT_A3, 'aim3_bryois_8celltype_coloc_MDD_and_ANX.tsv')
if os.path.exists(out_f):
    df_res = pd.read_csv(out_f, sep='\t')
    top_genes = df_res[df_res['PP.H4'] >= 0.40]['gene'].unique().tolist()
    if not top_genes:
        top_genes = df_res.groupby('gene')['PP.H4'].max().sort_values(ascending=False).head(8).index.tolist()
    sub = df_res[df_res['gene'].isin(top_genes) & (df_res['trait'] == 'MDD')]
    piv = sub.pivot_table(index='gene', columns='cell_type', values='PP.H4', aggfunc='max').fillna(0.0)
    if not piv.empty:
        fig, ax = plt.subplots(figsize=(11, max(4, len(piv)*0.6)), dpi=300)
        im = ax.imshow(piv.to_numpy(), cmap='YlGn', vmin=0, vmax=1.0, aspect='auto')
        ax.set_xticks(np.arange(piv.shape[1]))
        ax.set_xticklabels(piv.columns, rotation=30, ha='right', fontweight='bold')
        ax.set_yticks(np.arange(piv.shape[0]))
        ax.set_yticklabels(piv.index, fontweight='bold')
        for i in range(piv.shape[0]):
            for j in range(piv.shape[1]):
                v = piv.iloc[i, j]
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', color='white' if v > 0.6 else 'black', fontweight='bold' if v >= 0.7 else 'normal', fontsize=8)
        plt.colorbar(im, ax=ax, label='Colocalization PP.H4 (MDD GWAS vs Cell-Type eQTL)')
        ax.set_title('Bryois 2022 8-Brain-Cell-Type cis-eQTL Colocalization with PGC MDD GWAS', loc='left', fontweight='bold', pad=10)
        plt.tight_layout()
        fig_p = os.path.join(DRIVE_OUT_A3, 'Figure_Aim3_Bryois_CellType_eQTL_Coloc.png')
        plt.savefig(fig_p, dpi=300, bbox_inches='tight')
        plt.show()
        print(f'  -> Saved Bryois heatmap to Drive: {fig_p}', flush=True)
